# Fine-tune Laya for Dhristi's fast path (Colab, large GPU)

This replaces the Kaggle notebook. On one 80 GB GPU it runs a small sweep instead of a single run: **4 full-encoder runs × 6 epochs** on 20,000 generated steps, with checkpoints at epochs 2, 4 and 6. That gives 12 candidates. Each candidate is scored on generated validation steps it has not seen. **Only the selected one** then runs on the benchmark (`fastpath_bench.py`).

**Runtime:** *Runtime → Change runtime type →* **A100** or **H100**, *High-RAM* on. The notebook adapts its settings to a 40 GB card (A100 40 GB, L4), so it does not need exactly 80 GB.

**Why the benchmark is not used to choose.** The 30 held-out benchmark cases are the only test of wording the model has not seen. If they picked the winner out of 12, the score would be inflated by the choice. The selection rule below is fixed before any run.

**Acceptance bar (Jev's, from `Docs/decisions/brain-cloud-models-jev.md`):** at 0.9 the checkpoint must act on no step that needs free text, and it must be right on every held-out step it answers. Weights are uploaded only if the benchmark meets this.

The data is synthetic and written by us, as the benchmark cases are. A pass shows the model learned this task's format on synthetic pages. It is not evidence about real sites.

Results are written to Google Drive under `MyDrive/dhristi-laya/`. If the runtime disconnects, re-run all cells: finished runs are skipped.

In [ ]:
# 1. GPU and settings that follow from it
import json, os, subprocess, sys, time, shutil
from pathlib import Path
import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 or H100"
GPU = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
BF16 = torch.cuda.is_bf16_supported()
print(f"{GPU} | {VRAM_GB:.0f} GB | bf16 {BF16} | RAM {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.0f} GB")

# Effective batch 32 in every case (the authors' 2xT4 recipe: micro-batch 4 x accumulation 4 x 2 GPUs).
if VRAM_GB >= 70:
    MICRO_BATCH, GRAD_ACCUM, GRAD_CKPT = 32, 1, "0"
elif VRAM_GB >= 35:
    MICRO_BATCH, GRAD_ACCUM, GRAD_CKPT = 16, 2, "0"
else:
    MICRO_BATCH, GRAD_ACCUM, GRAD_CKPT = 8, 4, "1"
AMP_DTYPE = "bf16" if BF16 else "fp16"
print(dict(MICRO_BATCH=MICRO_BATCH, GRAD_ACCUM=GRAD_ACCUM, GRAD_CKPT=GRAD_CKPT, AMP_DTYPE=AMP_DTYPE))

In [ ]:
# 2. Google Drive for results (checkpoints stay on the runtime's local disk; only the selected one is copied to Drive)
from google.colab import drive
drive.mount("/content/drive")
RUN_TAG = "colab-v01"
OUT = Path(f"/content/drive/MyDrive/dhristi-laya/{RUN_TAG}")
CKPTS = Path("/content/ckpts")
for p in (OUT / "logs", OUT / "validation", CKPTS):
    p.mkdir(parents=True, exist_ok=True)
print(OUT)

In [ ]:
# 3. Code and the base checkpoint
BRANCH = "claude/determined-babbage-mhi2zj"   # change to master once PR #41 is merged
!pip install -q "laya==0.3.22" protobuf safetensors huggingface_hub
!rm -rf /content/sih-2026 && git clone -q --depth 1 --branch $BRANCH https://github.com/francisreubenr-rvu/sih-2026 /content/sih-2026
REPO = Path("/content/sih-2026")
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("commit", COMMIT)

from huggingface_hub import snapshot_download
BASE = snapshot_download("convaiinnovations/laya", allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"])
print(BASE)

## Data

20,000 training steps and 2,000 validation steps from `gen_data.py`, using the v4 settings (one right answer per step; about 22% of steps need free text), with different seeds. Both are checked to share no task or label with the benchmark cases. Validation steps that are identical to a training step are dropped when scoring. A dry run found about 1,700 of 2,000 left.

In [ ]:
# 4. Generate (kept on Drive so a rerun uses the same files)
TRAIN, VAL = OUT / "train20k.jsonl", OUT / "val2k.jsonl"
for path, n, seed in ((TRAIN, 20000, 20261010), (VAL, 2000, 20261011)):
    if not path.exists():
        out = subprocess.check_output([sys.executable, str(REPO / "scripts/laya-finetune/gen_data.py"), "--n", str(n), "--seed", str(seed), "--out", str(path)], text=True)
        report = json.loads(out)
        assert not report["shared_tasks"] and not report["shared_labels"], report
        print(path.name, {k: report[k] for k in ("examples", "free_text_true", "escape_gold")})
    else:
        print(path.name, "exists")

## Sweep (fixed before any run)

All runs train the whole encoder (28 of 28 layers), at 1024 / 256 tokens and an effective batch of 32, with a cosine schedule over 6 epochs. Checkpoints are saved at epochs 2, 4 and 6.

| Run | Encoder LR | Head LR | Seed |
|---|---|---|---|
| `a-s1` | 2.5e-5 | 1e-4 | 1 (the authors' rates) |
| `a-s2` | 2.5e-5 | 1e-4 | 2 (same rates, another seed: how much is noise) |
| `b-s1` | 5e-5 | 2e-4 | 1 |
| `c-s1` | 1e-5 | 5e-5 | 1 |

On an 80 GB A100 this is estimated at 2-3 hours from FLOP counts. That has not been measured, and an H100 should be roughly twice as fast. To spend less, drop runs from `RUNS`.

In [ ]:
# 5. Train
RUNS = {
    "a-s1": dict(LR_ENCODER="2.5e-5", LR_HEAD="1e-4", SEED="1"),
    "a-s2": dict(LR_ENCODER="2.5e-5", LR_HEAD="1e-4", SEED="2"),
    "b-s1": dict(LR_ENCODER="5e-5", LR_HEAD="2e-4", SEED="1"),
    "c-s1": dict(LR_ENCODER="1e-5", LR_HEAD="5e-5", SEED="1"),
}
COMMON = dict(EPOCHS="6", SAVE_EPOCHS="2,4", TRAIN_TOP_LAYERS="28", MAX_LEN="1024", HEAD_MAX_LEN="256",
              MICRO_BATCH=str(MICRO_BATCH), GRAD_ACCUM=str(GRAD_ACCUM), GRAD_CKPT=GRAD_CKPT, AMP_DTYPE=AMP_DTYPE)


def stream(cmd, env, log_path):
    """Run cmd, print its output live and keep a copy on Drive."""
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, env={**os.environ, **env}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
        for line in proc.stdout:
            if "warn" in line.lower():
                continue
            print(line, end="")
            log.write(line)
            log.flush()
        if proc.wait() != 0:
            raise RuntimeError(f"{cmd[1]} failed; see {log_path}")


def candidates(run):
    root = CKPTS / run
    return {f"{run}-e{e}": (root if e == 6 else root / f"epoch-{e}") for e in (2, 4, 6)}


def validated(run):
    return all((OUT / "validation" / f"{c}.json").exists() for c in candidates(run))


for run, env in RUNS.items():
    if (CKPTS / run / "training_meta.json").exists() or validated(run):
        print("skip", run, "(trained earlier)")
        continue
    t0 = time.time()
    stream([sys.executable, "scripts/laya-finetune/train_cpu.py", BASE, str(TRAIN), str(CKPTS / run)], {**COMMON, **env}, OUT / "logs" / f"train-{run}.log")
    print(f"=== {run} trained in {(time.time() - t0) / 60:.0f} min")

## Score every candidate on the unseen validation steps

The scoring goes through `warden/fastpath.decide`, which is how the Warden asks Laya at run time: compact keys, the free-text gate at 0.5, and the confidence threshold. A flat free-text AUROC near 0.5, with the same probability on every step, is the CPU failure mode.

In [ ]:
# 6. Validate
for run in RUNS:
    for cand, path in candidates(run).items():
        dest = OUT / "validation" / f"{cand}.json"
        if dest.exists():
            continue
        if not (path / "model.safetensors").exists():
            print("missing checkpoint", cand, "(trained before a disconnect?); it is left out")
            continue
        stream([sys.executable, "scripts/laya-finetune/validate.py", str(path), str(VAL), "--train", str(TRAIN), "--out", str(dest)],
               {"WARDEN_LAYA_DEVICE": "cuda"}, OUT / "logs" / f"validate-{cand}.log")

rows = []
for f in sorted((OUT / "validation").glob("*.json")):
    v = json.loads(f.read_text())
    g = v["by_threshold"]["0.9"]
    rows.append({"candidate": f.stem, "passes": v["passes_at_0.9"], "acted": g["acted"], "wrong": g["wrong"],
                 "acted_on_free_text": g["acted_on_free_text"], "free_text_auroc": v["free_text"]["auroc"],
                 "top_choice": f'{v["top_choice_right"]}/{v["steps"]}', "steps": v["steps"]})
import pandas as pd
pd.DataFrame(rows).set_index("candidate")

## Select (rule fixed in advance)

1. Keep the candidates that pass at 0.9 on validation: they act at least once, are wrong on none, and act on no free-text step.
2. Among those, take the one that acts on the most steps. Ties go to the higher free-text AUROC, then to fewer epochs.
3. If none passes, take the one with the fewest wrong answers at 0.9, then the most steps acted on. It is benchmarked so the failure is on record, and it is **not** accepted.

In [ ]:
# 7. Select, and retrain the winner if its checkpoint was lost to a disconnect
def epoch_of(c): return int(c.rsplit("-e", 1)[1])
passing = [r for r in rows if r["passes"]]
if passing:
    win = sorted(passing, key=lambda r: (-r["acted"], -(r["free_text_auroc"] or 0), epoch_of(r["candidate"])))[0]
else:
    win = sorted(rows, key=lambda r: (r["wrong"] + r["acted_on_free_text"], -r["acted"]))[0]
WIN = win["candidate"]
WIN_RUN = WIN.rsplit("-e", 1)[0]
WIN_PATH = candidates(WIN_RUN)[WIN]
print("selected", WIN, "| passes validation:", win["passes"])
if not (WIN_PATH / "model.safetensors").exists():
    print("checkpoint lost; retraining", WIN_RUN)
    stream([sys.executable, "scripts/laya-finetune/train_cpu.py", BASE, str(TRAIN), str(CKPTS / WIN_RUN)], {**COMMON, **RUNS[WIN_RUN]}, OUT / "logs" / f"train-{WIN_RUN}-again.log")
json.dump({"selected": WIN, "passed_validation": win["passes"], "candidates": rows, "rule": "see notebook section 'Select'",
           "commit": COMMIT, "gpu": GPU}, open(OUT / "selection.json", "w"), indent=1)

## Benchmark the selected checkpoint (once)

This uses the 12 design and 30 held-out cases. The LLM answers are reused from the committed run, so no Groq key is needed. Laya is deterministic, so one rep per case is enough.

In [ ]:
# 8. Benchmark
BENCH = OUT / "fastpath-bench-laya-colab-v01.json"
stream([sys.executable, "scripts/cloud-models/fastpath_bench.py", "--backends", "laya", "--laya-model", str(WIN_PATH), "--reps", "1",
        "--llm-from", "Benchmarks/results/fastpath-bench-v01.json", "--out", str(BENCH)], {"WARDEN_LAYA_DEVICE": "cuda"}, OUT / "logs" / "bench.log")
doc = json.loads(BENCH.read_text())
name, b = next(iter(doc["backends"].items()))
h9 = b["summary"]["heldout"]["0.9"]
print(json.dumps({"design@0.9": b["summary"]["design"]["0.9"], "heldout@0.9": h9}, indent=1))
print("\nfree-text probability per held-out case (should be high only on the free-text cases):")
for cid, c in b["cases"].items():
    if c["split"] == "heldout":
        r = c["rows"][0]
        print(f'  {"FREE " if c["needs_llm"] else "     "}{cid:28s} {r.get("freeTextProbability") or 0:.2f}  choice {r.get("choice")} @ {r.get("confidence") or 0:.2f}  {"right" if r.get("correct") else "wrong"}')
ACCEPTED = bool(win["passes"] and h9["fast_path_used"] > 0 and h9["fast_path_precision"] == 1.0 and not h9["wrong_on_needs_llm"])
print("\nACCEPTED (Jev's bar at 0.9):", ACCEPTED)
doc["model_training"] = {"script": "scripts/laya-finetune/train_cpu.py", "notebook": "scripts/laya-finetune/colab_train.ipynb",
                         "commit": COMMIT, "gpu": GPU, "selected": WIN, "selection": "selection.json (validation only; benchmark not used to choose)",
                         **json.loads((WIN_PATH / "training_meta.json").read_text())}
doc["accepted_at_0.9"] = ACCEPTED
cpu_scope = "Laya ran on this container's CPU (4 cores, no GPU)"
assert cpu_scope in doc["scope"]
doc["scope"] = doc["scope"].replace(cpu_scope, f"Laya ran on a Colab {GPU}")
BENCH.write_text(json.dumps(doc, indent=1) + "\n")

## Keep the weights (only if accepted)

The selected checkpoint (about 0.8 GB) is copied to Drive in every case. It is pushed to a **private** Hugging Face repo only if it met the bar. Add `HF_TOKEN` (write access, account `francisreubenr`) under the key icon in Colab's left sidebar and allow this notebook to use it.

In [ ]:
# 9. Save
dest = OUT / "selected-checkpoint"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(WIN_PATH, dest, ignore=shutil.ignore_patterns("epoch-*"))
print("copied to", dest)

if ACCEPTED:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    repo = "francisreubenr/laya-dhristi-fastpath"
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(dest), repo_id=repo, commit_message=f"{WIN} from {RUN_TAG} at {COMMIT[:7]}")
    print("uploaded to", repo, "(private)")
else:
    print("not accepted: nothing uploaded")
print("\nResults for the repo are in", OUT, ": selection.json, validation/*.json, fastpath-bench-laya-colab-v01.json, logs/")